In [ ]:
import pandas

file_name = 'debian_data.csv'

data = pandas.read_csv(file_name)

print(data.columns)

In [ ]:
len(data['hash'])

In [ ]:
data[:1]

In [ ]:
data_json = data.to_json(orient='records')

In [ ]:
import json

data_array = json.loads(data_json)

In [ ]:
examples = {}

for i, example in enumerate(data_array):
    
    file_key = example['project'] + ' ' + example['filename']
    if file_key not in examples:
        examples[file_key] = {}
    label = example['label']
    if label not in examples[file_key]:
        examples[file_key][label] = []
    examples[file_key][label].append(example)
    if i % 5000 == 4999:
        print(file_key)
    

In [ ]:
diff_usable = []
correct_counter = 0
for key in examples.keys():
    if len(examples[key][True]) != 1 or len(examples[key][False]) != 1:
        pass
    else:
        examples[key][True] = examples[key][True][0]
        examples[key][False] = examples[key][False][0]
        correct_counter += 1
        diff_usable.append(examples[key])
print(correct_counter)
print(len(diff_usable))


In [ ]:
#                                                          -8228664527580018723                      

json_file = open('debian_chrome_data_full.json', 'w')
json_file.write(data_json + '\n')
json_file.close()

json_file = open('debian_chrome_data_one-to-one-map.json', 'w')
json_file.write(json.dumps(diff_usable))
json_file.close()


In [ ]:
import json 

json_file = open('debian_chrome_data_one-to-one-map.json')
diff_usable = json.load(json_file)
json_file.close()

import re

pattern1 = re.compile("^[0-9]+ ")
pattern2 = re.compile("^[0-9]+$")

strings = ["   this is 123 good", '123 hello world 123', '   56 /* dsjkfk*/  ', '  47 ']
for s in strings:
    m1 = pattern1.match(s.strip())
    m2 = pattern2.match(s.strip())
    if m1 is not None:
        index = s.find(m1.group())
        l = len(m1.group())
        ms = s[(index + l):]
        print(ms, len(ms))
    elif m2 is not None:
        index = s.find(m2.group())
        l = len(m2.group())
        ms = s[(index + l):]
        print(ms, len(ms))

        
 

In [ ]:
diff_usable[0]

In [ ]:
def process_code(code):
    code_lines = code.split('\n')
    modified_code_lines = []
    for line in code_lines:
        trimmed_line = line.strip()
        m1 = pattern1.match(trimmed_line)
        m2 = pattern2.match(trimmed_line)
        if m1 is None and m2 is None:
            modified_code_lines.append(line)
        elif m1 is not None:
            matched_string = m1.group()
            index = line.find(matched_string)
            l = len(matched_string)
            trimmed_line = line[(index + l):]
            modified_code_lines.append(trimmed_line)
        elif m2 is not None:
            matched_string = m2.group()
            index = line.find(matched_string)
            l = len(matched_string)
            trimmed_line = line[(index + l):]
            modified_code_lines.append(trimmed_line)
    return '\n'.join(modified_code_lines)

In [ ]:
import os
import sys
import difflib

if not os.path.exists('seperate_files'):
    os.mkdir('seperate_files')

if not os.path.exists('seperate_files/previous'):
    os.mkdir('seperate_files/previous')
    
if not os.path.exists('seperate_files/next'):
    os.mkdir('seperate_files/next')

count = 0
    
for i, data in enumerate(diff_usable):
    print(i)
    previous_code = data['true']['code']
    next_code = data['false']['code']
    if previous_code is None or next_code is None:
        continue
    if len(previous_code.strip()) == 0 or len(next_code.strip()) == 0:
        continue
    with open(os.path.join('seperate_files/previous', str(i + 1) + '.c'), 'w') as fp:
        p = process_code(previous_code)
#         p = previous_code
        fp.write(p)
        fp.close()
        
    with open(os.path.join('seperate_files/next', str(i + 1) + '.c'), 'w') as fp:
        p = process_code(next_code)
#         p = next_code
        fp.write(p)
        fp.close()
    count += 1

In [ ]:
print(next_code)


In [ ]:
import difflib
import clang.cindex
import clang.enumerations
import csv

# set the config
clang.cindex.Config.set_library_path("/usr/lib/x86_64-linux-gnu")
clang.cindex.Config.set_library_file('/usr/lib/x86_64-linux-gnu/libclang-6.0.so.1')

previous_code = '1\n2\n3\n4'
next_code = '4\n3\n2\n1'

In [ ]:
pc = previous_code.split('\n')
nc = next_code.split('\n')
print(len(pc), len(nc))

In [ ]:
difflib.context_diff(pc, nc)

In [ ]:
import re
nlp = re.compile("-\d+,")
plp = re.compile("\+\d+,")
line = "@@ -48,6 +41,14 @@"
nl = nlp.search(line).group()
pl = plp.search(line).group()


# print(int(nl[1:-1]), int(pl[1:-1]))
def parse_line_numbers(line):
    assert line.find("@@") != -1
    nl = nlp.search(line).group()
    pl = plp.search(line).group()
    return int(pl[1:-1]), int(nl[1:-1])

print(parse_line_numbers(line))
    
    
    
for x in [s for s in difflib.unified_diff(pc, nc)]:
    print('d\t' , x)

In [ ]:
def get_added_and_deleted_line_numbers(pc, nc):
    if isinstance(pc, str):
        pc = pc.split('\n')
    if isinstance(nc, str):
        nc = nc.split('\n')
    diffs = [s for s in difflib.unified_diff(pc, nc)]
    if len(diffs) == 0:
        return [], []
    added = []
    deleted = []
    diffs = diffs[2:]
    cal = 0
    cdl = 0
    for i in range(len(diffs)):
        dl = diffs[i]
        if dl.startswith('@@'):
            cal, cdl = parse_line_numbers(dl)
#             print(cal, cdl, '\t found for\t', dl.strip())
#             print('-------------------------------------------')
        elif not dl.startswith('+') and not dl.startswith('-'):
#             print('Found\t', dl)
#             print('Current lns\t', cal, cdl)
#             print('Moving on')
#             print('============================================')
            cal += 1
            cdl += 1
        elif dl.startswith('+'):
#             print('Found Adding Line \t', dl)
#             print('Current lns\t', cal)
#             print('Adding to the addtions')
#             print('Moving On')
#             print('********************************************')
            added.append(cal)
            cal += 1
        elif dl.startswith('-'):
#             print('Found Deletions Line \t', dl)
#             print('Current lns\t', cdl)
#             print('Adding to the Deletions')
#             print('Moving On')
#             print('#############################################')
            deleted.append(cdl)
            cdl += 1
    return added, deleted

a, b = get_added_and_deleted_line_numbers(pc, nc)
print(a)
print(b)
        

In [ ]:
def read_file(path):
    file_text = ''
    with open(path) as fp:
        for line in fp:
            file_text += line
    return file_text

nc = read_file("seperate_files/next/221.c")
pc = read_file("seperate_files/previous/221.c")
a, b = get_added_and_deleted_line_numbers(pc, nc)
print(a)
print(b)

In [ ]:
class Tokenizer:
    # creates the object, does the inital parse
    def __init__(self, path, tokenizer_type='original'):
        self.index = clang.cindex.Index.create()
        self.tu = self.index.parse(path)
        self.path = self.extract_path(path)
        self.symbol_table = {}
        self.symbol_count = 1
        self.tokenizer_type = tokenizer_type
    
    # To output for split_functions, must have same path up to last two folders
    def extract_path(self, path):
        return "".join(path.split("/")[:-2])

    # does futher processing on a literal token  
    def process_literal(self, literal):
        cursor_kind = clang.cindex.CursorKind
        kind = literal.cursor.kind
        #print literal.spelling
    
        if kind == cursor_kind.INTEGER_LITERAL:
            return literal.spelling
            #return ["NUM"]

        if kind == cursor_kind.FLOATING_LITERAL:
            return ["NUM"]

        if kind == cursor_kind.IMAGINARY_LITERAL:
            return ["NUM"]

        if kind == cursor_kind.STRING_LITERAL:
            return ["STRING"]

        if kind == cursor_kind.CHARACTER_LITERAL:
            return ["CHAR"]

        if kind == cursor_kind.CXX_BOOL_LITERAL_EXPR:
            return ["BOOL"]

        # catch all other literals
        return ["LITERAL"]
    
    # filters out unwanted punctuation    
    def process_puntuation(self, punctuation):
        spelling = punctuation.spelling          
        return [spelling]
    
    # further processes and identifier token    
    def process_ident(self, ident):
        # assert type in ['original', 'symbolic', 'flat'], 'Token type must be original, symbolic, flat'
        # are we a "special" ident?
        # if ident.spelling in ["std", "cout", "cin", "vector", "pair", "string", "NULL", "size_t"]:
        return [ident.spelling]
#         ident_name = ident.spelling
#         if self.tokenizer_type == 'original':
#             return [ident_name]
#         elif self.tokenizer_type == 'symbolic':
#             if ident_name not in self.symbol_table:
#                 symbolic_name = 'ID' + str(self.symbol_count)
#                 self.symbol_table[ident_name] = symbolic_name
#                 self.symbol_count += 1
#             return [self.symbol_table[ident_name]]
#         # catch all others
#         else:
#             return ["ID"]
    
    # tokenizes the contents of a specific cursor
    def full_tokenize_cursor(self, cursor):
        tokens = cursor.get_tokens()
        # return final tokens as a list
        result = []

        for token in tokens:
            if token.kind.name == "COMMENT":
                continue
    
            else:
                spelling = token.spelling
                spelling = re.sub('\s+', ' ', spelling)
                
#                 if spelling in [';', '{', '}']:
#                     spelling += '\n'
                result += [spelling]
                continue
    
#             if token.kind.name == "IDENTIFIER":
#                 result += self.process_ident(token)
#                 continue
    
#             if token.kind.name == "KEYWORD":
#                 result += [token.spelling]
    
        return result
    
    def extract_line_numbers(self, cursor):
        tokens = cursor.get_tokens()
        lines = []
        for token in tokens:
            lines.append(token.location.line)
        return list(sorted(set(lines)))
    
    # tokenizes the entire document
    def full_tokenize(self):
        cursor = self.tu.cursor
        return self.full_tokenize_cursor(cursor)
    
    def extract_function_prototype(self, c):
        cursor_kind = clang.cindex.CursorKind        
        assert c.kind == cursor_kind.CXX_METHOD or c.kind == cursor_kind.FUNCTION_DECL
        name = c.spelling
        rtype = c.result_type.spelling
        args = []
        for arg in c.get_arguments():
            args.append(str(arg.type.spelling) + " " + str(arg.spelling))
        return str(rtype) + " " + str(name) + " (" + ', '.join(args) + ")"
    
    # returns a list of function name / function / filename tuples
    
    def extract_method_and_line_numbers(self):
        results = {}
        cursor_kind = clang.cindex.CursorKind
        
        cursor = self.tu.cursor
        for c in cursor.get_children():
            if c.kind == cursor_kind.CXX_METHOD or c.kind == cursor_kind.FUNCTION_DECL:
                name = self.extract_function_prototype(c)
                lines = self.extract_line_numbers(c)
                tokens = ' '.join(self.full_tokenize_cursor(c))
                tokens = re.sub('\s+', ' ', tokens)
                tokens = re.sub('{', '{\n', tokens)
                tokens = re.sub(';', ';\n', tokens)
                tokens = re.sub('}', '}\n', tokens)
                results[name] = (lines, tokens)
                
        return results
    
    def split_functions(self, method_only):
        results = []
        cursor_kind = clang.cindex.CursorKind
        
        # query all children for methods, and then tokenize each
        cursor = self.tu.cursor
        for c in cursor.get_children():
            filename = c.location.file.name if c.location.file != None else "NONE"
            extracted_path = self.extract_path(filename)
            
            if (c.kind == cursor_kind.CXX_METHOD or (method_only == False and c.kind == cursor_kind.FUNCTION_DECL)) and extracted_path == self.path:
                name = c.spelling
#                 tokens = self.full_tokenize_cursor(c)
#                 filename = filename.split("/")[-1]
#                 results += [tokens]
                lines = self.extract_line_numbers(c)

                print(self.extract_function_prototype(c), '\t->\n', lines)
                
        return results
    
f1 = "/home/saikatc/DATA/vulnerable-code/code-slicer/test/test1.c"
f2 = "seperate_files/next/221.c"
tok = Tokenizer(f2)
print(tok.extract_method_and_line_numbers())

In [ ]:
# for i, data in enumerate(diff_usable):
#     if i % 3 == 2:
#         break
#     previous_code = data[True]['code']
#     next_code = data[False]['code']
#     if previous_code is None or next_code is None:
#         continue
#     if len(previous_code.strip()) == 0 or len(next_code.strip()) == 0:
#         continue
        
#     pc = process_code(previous_code)
#     nc = process_code(next_code)
#     pfp = os.path.join('seperate_files/previous', str(i + 1) + '.c')
#     nfp = os.path.join('seperate_files/next', str(i + 1) + '.c')
# #     with open(pfp, 'w') as fp:
# #         fp.write(pc)
# #         fp.close()
        
# #     with open(nfp, 'w') as fp:
# #         fp.write(nc)
# #         fp.close()
#     count += 1


def compressed_text(text):
    return text.replace(' ', '').replace('\n', '').replace('\t', '')

def extract_selected_lines_from_file_text_string(fp_text, lines):
    fp_lines = fp_text.split('\n')
    return_text = ''
    for i in range(len(fp_lines)):
        if (i + 1) in lines:
            return_text += (fp_lines[i] + '\n')
    return return_text, compressed_text(return_text)

# print(extract_selected_lines_from_file_text_string(
#     "chrome connection_info_popup_android.cc\nchrome dtitv_impl.h\n" + \
#     "chrome CanvasAsyncBlobCreator.h\nchrome node_channel.cc" + \
#     "\nchrome child_process_security_policy_impl.cc\nchrome pngconf.h\nchrome gles2_cmd_decoder_unittest_base.cc",
#     [2,4]
# ))


In [ ]:
import json

json_file = open('debian_chrome_data_one-to-one-map.json')
diff_usables = json.load(json_file)
json_file.close()



In [ ]:
import os, sys

vulnerable_functions = []
non_vulnerable_functions = []
vulnerable_num_lines = []
non_vulnerable_num_lines = []

vulnerable_set = set()
non_vulnerable_set = set()

for i in range(len(diff_usables)):
    if i % 25 == 24:
        print(i, len(vulnerable_functions), len(non_vulnerable_functions))
    pfp = os.path.join('seperate_files/previous', str(i + 1) + '.c')
    nfp = os.path.join('seperate_files/next', str(i + 1) + '.c')
    if not os.path.exists(pfp) or not os.path.exists(nfp):
        continue
    pc = read_file(pfp)
    nc = read_file(nfp)
    
    pdp = diff_usables[i]['true']
    ndp = diff_usables[i]['false']
    
    prev_tokenizer = Tokenizer(pfp)
    previous_version_method_lines = prev_tokenizer.extract_method_and_line_numbers()
    
    next_tokenizer = Tokenizer(nfp)
    next_version_method_lines = next_tokenizer.extract_method_and_line_numbers()
    
    previous_version_method_names = set(previous_version_method_lines.keys())
    next_version_method_names = set(next_version_method_lines.keys())
    
    common_methods = previous_version_method_names.intersection(next_version_method_names)
    only_previous = previous_version_method_names.difference(next_version_method_names)
    only_next = next_version_method_names.difference(previous_version_method_names)
    
    added_lines, deleted_lines = get_added_and_deleted_line_numbers(pc, nc)
#     for function_prototype in only_next:
#         lines, text = next_version_method_lines[function_prototype]
#         # text, ctext = extract_selected_lines_from_file_text_string(nc, lines)
#         ctext = compressed_text(text)
#         if len(text.strip()) == 0:
#                 continue
#         if ctext not in non_vulnerable_set and len(text.strip().split('\n')) > 1:
#             non_vulnerable_functions.append(
#                 {
#                     'code': text.strip(),
#                     'hash': ndp['hash'],
#                     'project': ndp['project'],
#                     'size': len(text.strip().split('\n'))
                    
#                 }
#             )
#             non_vulnerable_set.add(ctext)
#             non_vulnerable_num_lines.append(len(lines))
        
    for function_prototype in only_previous:
        lines, text = previous_version_method_lines[function_prototype]
        # text, ctext = extract_selected_lines_from_file_text_string(pc, lines)
        ctext = compressed_text(text)
        if len(text.strip()) == 0:
                continue
        if ctext not in vulnerable_set and len(text.strip().split('\n')) > 1:    
            vulnerable_functions.append(
                {
                    'code': text.strip(),
                    'hash': pdp['hash'],
                    'project': pdp['project'],
                    'size': len(text.strip().split('\n'))
                }
            )
            vulnerable_set.add(ctext)
            vulnerable_num_lines.append(len(lines))
    
    for function_prototype in common_methods:
        previous_version_lines, prev_text = previous_version_method_lines[function_prototype]
        next_version_lines, next_text = next_version_method_lines[function_prototype]
        lines_deleted_from_prev = set(previous_version_lines).intersection(set(deleted_lines))
        lines_added_to_next = set(next_version_lines).intersection(set(added_lines))
        if len(lines_added_to_next) > 0 or len(lines_deleted_from_prev) > 0:
            # text, ctext = extract_selected_lines_from_file_text_string(pc, previous_version_lines)
            text = prev_text
            ctext = compressed_text(text)
            if len(text.strip()) == 0:
                continue
            if ctext not in vulnerable_set and len(text.strip().split('\n')) > 1:
                vulnerable_functions.append(
                    {
                        'code': text.strip(),
                        'hash': pdp['hash'],
                        'project': pdp['project'],
                        'size': len(text.strip().split('\n'))

                    }
                )
                vulnerable_set.add(ctext)
                vulnerable_num_lines.append(len(previous_version_lines))
                
            # text, ctext = extract_selected_lines_from_file_text_string(nc, next_version_lines)
#             text = next_text
#             ctext = compressed_text(text)
#             if len(text.strip()) == 0:
#                 continue
#             if ctext not in non_vulnerable_set and len(text.strip().split('\n')) > 1:
#                 non_vulnerable_functions.append(
#                     {
#                         'code': text.strip(),
#                         'hash': ndp['hash'],
#                         'project': ndp['project'],
#                         'size': len(text.strip().split('\n'))
#                     }
#                 )
#                 non_vulnerable_set.add(ctext)
#                 non_vulnerable_num_lines.append(len(next_version_lines))
        else:
            # text, ctext = extract_selected_lines_from_file_text_string(pc, previous_version_lines)
            text = prev_text
            ctext = compressed_text(text)
            if len(text.strip()) == 0:
                continue
            if ctext not in non_vulnerable_set and len(text.strip().split('\n')) > 1:
                non_vulnerable_functions.append(
                    {
                        'code': text.strip(),
                        'hash': pdp['hash'],
                        'project': pdp['project'],
                        'size': len(text.strip().split('\n'))
                    }
                )
                non_vulnerable_set.add(ctext)
                non_vulnerable_num_lines.append(len(previous_version_lines))
                
print(vulnerable_functions[0])
print(non_vulnerable_functions[0])

print(vulnerable_functions[1]['code'])
print(non_vulnerable_functions[1]['code'])
import json
vjson = open('vulnerables-vertical.json', 'w')
vjson.write(json.dumps(vulnerable_functions))
vjson.close()


nvjson = open('non-vulnerables-vertical.json', 'w')
nvjson.write(json.dumps(non_vulnerable_functions))
nvjson.close()
